In [2]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Pour le modèle LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Chargement des données
# Note: Téléchargez d'abord le fichier depuis: https://archive.ics.uci.edu/ml/datasets/individual+household+electric+power+consumption
data_path = '/content/household_power_consumption.txt'  # Ajustez le chemin selon votre téléchargement

try:
    df = pd.read_csv(data_path, sep=';',
                     parse_dates={'datetime': ['Date', 'Time']},
                     infer_datetime_format=True,
                     low_memory=False,
                     na_values=['?'])
except FileNotFoundError:
    # Alternative: télécharger directement depuis UCI
    import urllib.request
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
    import zipfile
    import io

    print("Téléchargement du dataset...")
    with urllib.request.urlopen(url) as response:
        with zipfile.ZipFile(io.BytesIO(response.read())) as zip_file:
            with zip_file.open('household_power_consumption.txt') as file:
                df = pd.read_csv(file, sep=';',
                               parse_dates={'datetime': ['Date', 'Time']},
                               infer_datetime_format=True,
                               low_memory=False,
                               na_values=['?'])

# Affichage des premières lignes
print("Premières lignes du dataset:")
print(df.head())

print("\nStructure du dataset:")
print(df.info())

print("\nStatistiques descriptives:")
print(df.describe())

print(f"\nNombre total d'enregistrements: {len(df)}")
print(f"Période: de {df['datetime'].min()} à {df['datetime'].max()}")
print(f"Nombre de colonnes: {len(df.columns)}")

# Vérification du type de données de chaque colonne
print("\nTypes de données par colonne:")
print(df.dtypes)

# Colonnes du dataset (après conversion)
# datetime, Global_active_power, Global_reactive_power, Voltage,
# Global_intensity, Sub_metering_1, Sub_metering_2, Sub_metering_3

Téléchargement du dataset...
Premières lignes du dataset:
             datetime  Global_active_power  Global_reactive_power  Voltage  \
0 2006-12-16 17:24:00                4.216                  0.418   234.84   
1 2006-12-16 17:25:00                5.360                  0.436   233.63   
2 2006-12-16 17:26:00                5.374                  0.498   233.29   
3 2006-12-16 17:27:00                5.388                  0.502   233.74   
4 2006-12-16 17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  

Structure du dataset:
<class 'pandas.core.frame.Data